In [ ]:
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
import cv2
import cv2
import numpy as np
from typing import Tuple
from pathlib import Path

In [2]:
def get_player_from_centroid_position(centroid: Tuple[int], image_shape: Tuple[int, int]) -> int:
    dx = (centroid[0] - image_shape[1] / 2.0) / image_shape[1]
    dy = (centroid[1] - image_shape[0] / 2.0) / image_shape[0]
    if abs(dx) > abs(dy):
        return 2 if dx > 0 else 4    # right or left
    return 1 if dy > 0 else 3        # bottom or top

In [ ]:
def detect_active_player_yellow(image: np.ndarray) -> int:
    """
    Find the yellow active-player token in an Uno game image.

    Returns 1 if the token is on the bottom, 2 if right, 3 if top, 4 if left. Returns 0 if no token is found.
    """
    # Create an hsv mask to isolate strong candidates for the yellow token
    hsv = cv2.cvtColor(image, cv2.COLOR_RGB2HSV)
    mask = cv2.inRange(hsv, np.array([18, 70, 235]), np.array([29, 255, 255]))

    # Clean the mask by removing small objects and filling small holes
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (15, 15))
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel)
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel)

    # Find the yellow blob whose contour shape and area is most close to the token's
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL,
                                   cv2.CHAIN_APPROX_SIMPLE)
    best_center, best_circ = None, 0.7

    for cnt in contours:
        area = cv2.contourArea(cnt)
        if not (15000 < area < 50000):  # Relatively big range to remove outliers without risking to eliminate the token
            continue
        circularity = 4 * np.pi * area / cv2.arcLength(cnt, True) ** 2
        if circularity < best_circ:
            continue
        x, y, w, h = cv2.boundingRect(cnt)
        fill = cv2.countNonZero(mask[y:y+h, x:x+w]) / area
        if fill < 0.8:  # Reject rings that are not filled enough (specifically meant to avoid "skip turn" cards)
            continue
        M = cv2.moments(cnt)
        best_center = (M['m10'] / M['m00'], M['m01'] / M['m00'])
        best_circ = circularity

    if best_center is None:
        return 0

    # Find the player from the center of the token's identified shape
    return get_player_from_centroid_position(best_center, image.shape[:2])

In [ ]:
def detect_active_player_black(image: np.ndarray) -> int:
    """
    Find the black active-player token in an Uno game image.

    Returns 1 if the token is on the bottom, 2 if right, 3 if top, 4 if left. Returns 0 if no token is found.
    """
    # Create an hsv mask to isolate strong candidates for the black token
    hsv = cv2.cvtColor(image, cv2.COLOR_RGB2HSV)
    mask = cv2.inRange(hsv, np.array([0, 0, 0]), np.array([179, 55, 150]))

    # Clean the mask by removing small objects and filling small holes
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (13, 13))
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel)
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel)

    # Find the black blob whose contour shape and area is most close to the token's
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL,
                                   cv2.CHAIN_APPROX_SIMPLE)
    best_center, best_score = None, -1.0

    for cnt in contours:
        area = cv2.contourArea(cnt)
        if area < 5000:                    # skip pure noise
            continue
        (cx, cy), (w, h), _ = cv2.minAreaRect(cnt)
        if w == 0 or h == 0:
            continue
        rect_score = area / (w * h)        # 1.0 = perfect rectangle
        ratio = max(w, h) / min(w, h)

        # Define a score where 1 means we are at the approximate ideal value for each and keep the best
        area_fit = 1 - min(1, abs(area - 40000) / 40000)
        aspect_fit = 1 - min(1, abs(ratio - 1.6) / 1.6)
        score = area_fit * rect_score * aspect_fit

        if score > best_score:
            best_score = score
            best_center = (cx, cy)

    if best_center is None:
        return 0

    # Find the player from the center of the token's identified shape
    return get_player_from_centroid_position(best_center, image.shape[:2])

In [5]:
def detect_background(image: np.ndarray) -> str:
    """Return which image background type the image has."""
    hsv = cv2.cvtColor(image, cv2.COLOR_RGB2HSV)
    return "leafy" if hsv[:, :, 1].mean() > 50 else "white"

In [ ]:
def detect_active_player(image: np.ndarray) -> int:
    """Output the current active player based on the image."""
    if detect_background(image) == "leafy":
        active_player = detect_active_player_yellow(image)
    else:
        active_player = detect_active_player_black(image)
    if active_player == 0:  # If no token was found, assign it to player 1 as a default
        return 1
    
    return active_player

In [ ]:
IMAGE_DIR = Path("../data/iapr-26-uno-vision-challenge/train_images")   # <-- edit this
SIDE_NAMES = {1: "bottom", 2: "right", 3: "top", 4: "left"}

image_paths = sorted(p for p in IMAGE_DIR.iterdir()
                     if p.suffix.lower() in {".jpg", ".jpeg", ".png"})

for img_path in image_paths:
    rgb = np.array(Image.open(img_path))
    if rgb is None:
        print(f"{img_path.name}: failed to read")
        continue

    result = detect_active_player(rgb)

    plt.figure(figsize=(8, 6))
    plt.imshow(rgb)
    plt.title(f"{img_path.name}  →  player {result} ({SIDE_NAMES[result]})")
    plt.axis("off")
    plt.show()